In [9]:
import hail as hl
import os

from gnomad.utils.vep import process_consequences
from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [10]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

2021-11-08 14:46:12 Hail: WARN: Hail has already been initialized. If this call was intended to change configuration, close the session with hl.stop() first.


FatalError: IllegalArgumentException: requirement failed

Java stack trace:
java.lang.IllegalArgumentException: requirement failed
	at scala.Predef$.require(Predef.scala:268)
	at is.hail.backend.spark.SparkBackend$.apply(SparkBackend.scala:218)
	at is.hail.backend.spark.SparkBackend.apply(SparkBackend.scala)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:357)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.GatewayConnection.run(GatewayConnection.java:238)
	at java.lang.Thread.run(Thread.

In [73]:
path1 = 'data/mt/ukb_wes_200k_annotated_chr21.mt'
mt1 = hl.read_matrix_table(path1)

In [74]:
# load VEP annotation table
input_annotation_path = "data/vep/hail/ukb_wes_200k_chr21_vep.ht"
consequence_annotations = hl.read_table(input_annotation_path)
mt1 = mt1.annotate_rows(consequence = consequence_annotations[mt1.row_key]) 

In [75]:
mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
by_gene_annotation1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_by_gene_canonical, mt1.consequence.dbnsfp, use_loftee = False)
mt1 = mt1.annotate_rows(consequence_category = by_gene_annotation1) 
#mt1_subset = mt1.filter_rows(hl.literal(set(['ptv'])).contains(mt1.consequence_category)) 
mt1_subset = mt1.filter_rows(hl.literal(set(['damaging_missense'])).contains(mt1.consequence_category)) 



In [76]:
#mt1_subset.count()
#mt1 = mt1.filter_rows(mt1.locus.position == 14181782)

In [77]:
#mt1.count()

In [81]:
#print(mt1_subset.count())
ht = mt1_subset
ht = ht.drop('consequence')
out = hl.vep(ht, 'utils/configs/vep_final.json')
out = process_consequences(out)


In [90]:
#out.vep.describe()
#out.vep.transcript_consequences.lof.show()
#out.vep.transcript_consequences.cadd_phred.show()
#out.vep.transcript_consequences.revel_score.show()

out = out.explode_rows(out.vep.worst_csq_by_gene_canonical)
annotation_case_builder(out.vep.worst_csq_by_gene_canonical)

#out.vep.worst_csq_by_gene_canonical.revel_score.

NameError: name 'MISSENSE_CSQS' is not defined

In [88]:
def annotation_case_builder(worst_csq_expr, use_loftee: bool = True):
    r'''Annotate consequence categories for downstream analysis
    :param worst_csq_by_gene_canonical_expr: A struct that should contain "most_severe_consequence"
    :param use_loftee: if True will annotate PTVs as either high confidence (ptv) or low confidence (ptv_LC)

    '''
    case = hl.case(missing_false=True)
    if use_loftee:
        case = (case
                .when(worst_csq_expr.lof == 'HC', 'ptv')
                .when(worst_csq_expr.lof == 'LC', 'ptv_LC')
                )
    else:
        case = case.when(
            hl.set(PLOF_CSQS).contains(
                worst_csq_by_gene_canonical_expr.most_severe_consequence),
            "ptv")
    case = (case.when(hl.set(MISSENSE_CSQS).contains(worst_csq_expr.most_severe_consequence) &
                      (~hl.is_defined(worst_csq_expr.cadd_phred) | ~hl.is_defined(worst_csq_expr.revel_score)), "other_missense")
                .when(hl.set(MISSENSE_CSQS).contains(worst_csq_expr.most_severe_consequence) &
                      (worst_csq_expr.cadd_phred_score >= 20) & (worst_csq_expr.revel_score >= 0.6), "damaging_missense")
                .when(hl.set(MISSENSE_CSQS).contains(worst_csq_expr.most_severe_consequence), "other_missense")
                .when(hl.set(SYNONYMOUS_CSQS).contains(worst_csq_expr.most_severe_consequence), "synonymous")
                .when(hl.set(OTHER_CSQS).contains(worst_csq_expr.most_severe_consequence), "non_coding")
            )
    return case.or_missing()

In [ ]:
def gene_csqs_case_builder(in_mt):
    r''' Returns matrix table that contains gene consequence information from phased geneotypes.
    "": no alternate alleles,
    "HE": one alternate allele on either strand in a locus, 
    "HO": homozygous for alternate alleles
    "CH": two alternate allele on either strand in a locus (compound heterozygous)
    "CH+HO": two alternate allele on either strand in a locus (either as homozygous or compound heterozygous)
    '''
    # create one gene_id for each item in gene_id array
    #in_mt = in_mt.explode_rows(in_mt.vep.worst_csq_by_gene_canonical)
    # get all snps that are not homozygous
    mt = in_mt
    mt = analysis.annotate_phased_entries(mt)
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    # create table for each strand and combine to gene
    ht0 = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(s0 = hl.agg.any(mt.a0_alt)))
    ht1 = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(s1 = hl.agg.any(mt.a1_alt)))
    ht2 = (in_mt.group_rows_by(in_mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(hom_var = hl.agg.any(in_mt.GT.is_hom_var())))
    # combine entries
    ht = ht0.annotate_entries(s1 = ht1[ht0.gene_id, ht0.s].s1)
    ht = ht.annotate_entries(hom_var = ht2[ht.gene_id, ht.s].hom_var)
    expr = (hl.case()
           .when( (ht.s0) & (ht.s1) & (ht.hom_var), 'CH+HO')
           .when( (ht.s0) & (ht.s1), "CH")
           .when( (ht.hom_var), 'HO')
           .when( (ht.s0) & (ht.s1 == False), 'HE')
           .when( (ht.s1) & (ht.s0 == False), 'HE')
           .default(''))
    ht = ht.annotate_entries(csqs = expr)
    ht = ht.drop('s0').drop('s1').drop('hom_var')    
    return ht

In [ ]:
mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.dtype

In [ ]:
mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.dtype == hl.expr.ArrayExpression

In [ ]:
hl.expr.ArrayExpression(hl.str(''))